# Notebook 05: Color Detection

## Before You Start
> **If anything behaves unexpectedly, restart the kernel first: Kernel menu → Restart Kernel and Clear All Outputs. Then run the cells from the top.**

## ADAS Connection
Self-driving cars need to recognize specific things in their environment -- red stop signs, green traffic lights, yellow lane markings, orange construction cones. This is called **color-based object detection** and it is one of the foundational techniques in computer vision.

In this notebook you will teach your robot to recognize a specific color in real time. By the end, your robot will be able to find your team color anywhere in its camera view and tell you exactly where it is.

---

## How It Works
Remember from Notebook 04 that HSV separates **hue** (the actual color) from **brightness**. To detect a color we define a range of acceptable hue, saturation, and value -- called the **lower and upper bounds**. Any pixel that falls within that range gets marked as detected.

This produces a **mask** -- a black and white image where white means "this pixel is the target color" and black means "this pixel is not."

From the mask we can find **contours** -- the outlines of the detected color blobs. The largest contour is most likely our target object.

---

## The Code
Run this cell first to set up the camera and color ranges.

In [27]:
import cv2
import numpy as np
import ipywidgets as widgets
import threading
import time
from IPython.display import display

# ═══════════════════════════════════════
#   TWEAK THESE IF IMAGE IS TOO DARK
BRIGHTNESS = 80    # 0-100
CONTRAST   = 80    # 0-100
# ═══════════════════════════════════════
cap.set(cv2.CAP_PROP_BRIGHTNESS, BRIGHTNESS)
cap.set(cv2.CAP_PROP_CONTRAST,   CONTRAST)

def bgr8_to_jpeg(frame):
    return bytes(cv2.imencode('.jpg', frame)[1])

# Open the camera
try:
    cap.release()
except:
    pass

cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH,  640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
cap.set(cv2.CAP_PROP_FPS, 30)
cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter.fourcc('M','J','P','G'))
cap.set(cv2.CAP_PROP_BRIGHTNESS, BRIGHTNESS)   # try 20-60
cap.set(cv2.CAP_PROP_CONTRAST,   CONTRAST)   # try 20-60
cap.set(cv2.CAP_PROP_EXPOSURE,   100)  # try 50-200

# Color ranges in HSV
# Format: [H_min, S_min, V_min], [H_max, S_max, V_max]
COLOR_RANGES = {
    'red':    ([0,   43,  46],  [10,  255, 255]),
    'green':  ([35,  43,  46],  [77,  255, 255]),
    'blue':   ([100, 43,  46],  [124, 255, 255]),
    'yellow': ([26,  43,  46],  [34,  255, 255]),
    'orange': ([11,  43,  46],  [25,  255, 255]),
}

if ret:
    print('Camera ready!')
    print(f'Available colors: {list(COLOR_RANGES.keys())}')
    print()
    print('>>> Hold your team color object in front of the camera, then run the next cell.')
else:
    print('ERROR: Could not open camera.')

Camera ready!
Available colors: ['red', 'green', 'blue', 'yellow', 'orange']

>>> Hold your team color object in front of the camera, then run the next cell.


Now get the object with your team color and hold it in front of the camera.

In [28]:
# ═══════════════════════════════════════
#   HOLD YOUR TEAM COLOR OBJECT IN FRONT 
#   OF THE CAMERA BEFORE RUNNING THIS CELL
# ═══════════════════════════════════════
# Warm up the camera -- let it auto-adjust exposure
print('Warming up camera...')
for _ in range(20):
    cap.read()
    time.sleep(0.05)
    
ret, test_frame = cap.read()
if ret:
    # Show a preview so students know the camera is working
    preview_widget = widgets.Image(format='jpeg', width=640, height=480)
    display(preview_widget)
    preview_widget.value = bgr8_to_jpeg(test_frame)
    print('Frame captured! Now run Tweak Zone 1.')
else:
    print('ERROR: Could not open camera.')

Warming up camera...


Image(value=b'', format='jpeg', height='480', width='640')

Frame captured! Now run Tweak Zone 1.


---

## YOUR TURN -- Tweak Zone 1: Pick Your Team Color

Change `TARGET_COLOR` to your team color. Run the cell and hold your team color object in front of the camera.

The display shows three views side by side:
- **Left:** original camera feed
- **Center:** the mask -- white where your color is detected
- **Right:** result -- only your color visible, everything else black

> **Think like an engineer:** Why does the mask sometimes detect things you didn't expect? What else in the room might be close to your color's HSV range?

In [29]:
# ═══════════════════════════════════════
#   TWEAK THIS VALUE
TARGET_COLOR = 'red'    # try: red, green, blue, yellow, orange
# ═══════════════════════════════════════

color_lower = np.array(COLOR_RANGES[TARGET_COLOR][0])
color_upper = np.array(COLOR_RANGES[TARGET_COLOR][1])

ret, frame = cap.read()
if ret:
    hsv  = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, color_lower, color_upper)
    mask = cv2.erode(mask,  None, iterations=2)
    mask = cv2.dilate(mask, None, iterations=2)
    result = cv2.bitwise_and(frame, frame, mask=mask)
    
    mask_bgr = cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR)
    combined = np.hstack([frame, mask_bgr, result])
    
    cv2.putText(combined, 'Original',    (10,  30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,255), 2)
    cv2.putText(combined, 'Mask',        (650, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,255), 2)
    cv2.putText(combined, 'Result',      (1290,30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,255), 2)
    
    img_widget = widgets.Image(format='jpeg', width=1280, height=427)
    display(img_widget)
    img_widget.value = bgr8_to_jpeg(combined)
    print(f'Detecting: {TARGET_COLOR}')

Image(value=b'', format='jpeg', height='427', width='1280')

Detecting: red


---

## YOUR TURN -- Tweak Zone 2: Find the Blob

Start with MIN_RADIUS set to 200 -- nothing will be detected. 
Slowly lower the value and run the cell each time until the detection circle appears around your color object.

The radius tells you how large the detected blob is in pixels. 
Too small and you get false detections from noise. Too large and you miss real objects that are far away.

> **Think like an engineer:** How would you set MIN_RADIUS differently for detecting a stop sign vs a traffic cone? What about a pedestrian wearing a red jacket?

In [30]:
# ═══════════════════════════════════════
#   TWEAK THESE VALUES
MIN_RADIUS   = 200     # minimum blob size to count as detected (pixels)
CIRCLE_COLOR = (0, 255, 255)  # color of the detection circle (BGR)
# ═══════════════════════════════════════

ret, frame = cap.read()
if ret:
    hsv   = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    mask  = cv2.inRange(hsv, color_lower, color_upper)
    mask  = cv2.erode(mask,  None, iterations=2)
    mask  = cv2.dilate(mask, None, iterations=2)
    mask  = cv2.GaussianBlur(mask, (3,3), 0)
    cnts  = cv2.findContours(mask.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[-2]
    
    detected = False
    if len(cnts) > 0:
        cnt = max(cnts, key=cv2.contourArea)
        (cx, cy), radius = cv2.minEnclosingCircle(cnt)
        if radius > MIN_RADIUS:
            detected = True
            cv2.circle(frame, (int(cx), int(cy)), int(radius), CIRCLE_COLOR, 2)
            cv2.circle(frame, (int(cx), int(cy)), 5, CIRCLE_COLOR, -1)
            cv2.putText(frame, f'{TARGET_COLOR} detected', (int(cx)-60, int(cy)-int(radius)-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, CIRCLE_COLOR, 2)
            print(f'Detected {TARGET_COLOR} at position ({int(cx)}, {int(cy)}) radius={int(radius)}px')
    
    if not detected:
        cv2.putText(frame, 'Not detected', (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,0,255), 2)
        print(f'No {TARGET_COLOR} detected -- try holding the object closer or adjusting MIN_RADIUS')
    
    img_widget2 = widgets.Image(format='jpeg', width=640, height=480)
    display(img_widget2)
    img_widget2.value = bgr8_to_jpeg(frame)

Detected red at position (308, 239) radius=390px


Image(value=b'', format='jpeg', height='480', width='640')

---

## YOUR TURN -- Tweak Zone 3: Live Detection Feed

Now let's run detection continuously on a live feed. Move the colored object around and watch the detection circle follow it.

Change `SHOW_POSITION` to True to see the X/Y coordinates printed on screen -- this is the data we will use in the next notebook to control the motors.

> **Think like an engineer:** Notice the X position of the detected object. When X is less than 320 the object is on the left side of the frame. When X is greater than 320 it is on the right. How would you use this to steer a car?

In [23]:
# ═══════════════════════════════════════
#   TWEAK THESE VALUES
SHOW_POSITION = True    # True or False -- show X/Y on screen
MIN_RADIUS    = 20      # minimum blob size
# ═══════════════════════════════════════

feed_widget = widgets.Image(format='jpeg', width=640, height=480)
display(feed_widget)

running = True

def detect_feed():
    while running:
        ret, frame = cap.read()
        if not ret:
            break
        
        hsv  = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        mask = cv2.inRange(hsv, color_lower, color_upper)
        mask = cv2.erode(mask,  None, iterations=2)
        mask = cv2.dilate(mask, None, iterations=2)
        mask = cv2.GaussianBlur(mask, (3,3), 0)
        cnts = cv2.findContours(mask.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[-2]
        
        # draw center line
        cv2.line(frame, (320, 0), (320, 480), (255,255,255), 1)
        
        if len(cnts) > 0:
            cnt = max(cnts, key=cv2.contourArea)
            (cx, cy), radius = cv2.minEnclosingCircle(cnt)
            if radius > MIN_RADIUS:
                cv2.circle(frame, (int(cx), int(cy)), int(radius), (0,255,255), 2)
                cv2.circle(frame, (int(cx), int(cy)), 5, (0,255,255), -1)
                if SHOW_POSITION:
                    side = 'LEFT' if cx < 320 else 'RIGHT'
                    cv2.putText(frame, f'X:{int(cx)} Y:{int(cy)} -- {side}',
                                (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,255), 2)
        else:
            cv2.putText(frame, 'Searching...', (10, 40),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,255), 2)
        
        feed_widget.value = bgr8_to_jpeg(frame)
        time.sleep(0.033)

detect_thread = threading.Thread(target=detect_feed)
detect_thread.daemon = True
detect_thread.start()
print('Detection feed started. Run the STOP cell when done.')

Image(value=b'', format='jpeg', height='480', width='640')

Detection feed started. Run the STOP cell when done.


In [24]:
# STOP CELL -- run this to stop the feed
running = False
time.sleep(0.5)
print('Feed stopped.')

Feed stopped.


---

## What Happened?

Think about these questions with your team:

1. Did your color get detected reliably? What made it harder -- distance, lighting, background?
2. What happened when you held two objects of similar colors in front of the camera?
3. Look at the X position value. What number would mean the object is perfectly centered?
4. A self-driving car uses color detection for traffic lights. What challenges would it face at night? In fog?

---

## CHALLENGE -- Advanced Students

The HSV ranges in COLOR_RANGES are fixed values. But lighting conditions vary -- what works in this room might not work outside.

Write a calibration function that captures 10 frames, samples the HSV values of the center pixel, and automatically calculates the lower and upper bounds with some tolerance. This is called **adaptive thresholding**.

In [ ]:
# YOUR CODE HERE
# ═══════════════════════════════════════
TOLERANCE = 20    # how much variation to allow in each HSV channel
SAMPLES   = 10    # number of frames to sample
# ═══════════════════════════════════════

def calibrate_color(cap, samples=SAMPLES, tolerance=TOLERANCE):
    # Sample the center pixel across multiple frames
    # Calculate mean HSV values
    # Return lower and upper bounds with tolerance
    pass


---

## Always clean up when you are done!

In [31]:
running = False
time.sleep(0.5)
cap.release()
print('Camera released.')

Camera released.
